# Worker Node — seebotsia
Proxy + tarefas background no Colab.


In [ ]:
!pip install -q flask requests pyngrok


In [ ]:
# Worker Colab: proxy + tarefas backgroundimport subprocessimport osimport timeimport threadingdef install_deps():    """Instala dependências básicas."""    os.system("pip install -q flask requests pyngrok")def run_proxy(port=5000):    """Roda um proxy HTTP simples."""    from flask import Flask, request, Response    import requests as req    app = Flask(__name__)    @app.route('/proxy', methods=['GET', 'POST'])    def proxy():        url = request.args.get('url')        if not url:            return Response("Falta ?url=", status=400)        try:            r = req.get(url, timeout=30)            return Response(r.content, status=r.status_code,                          headers={'Content-Type': r.headers.get('Content-Type', 'text/html')})        except Exception as e:            return Response(str(e), status=500)    @app.route('/health')    def health():        return {"status": "ok", "uptime": time.time()}    app.run(host='0.0.0.0', port=port)def expose_ngrok(port=5000):    """Expõe o proxy via ngrok."""    from pyngrok import ngrok    tunnel = ngrok.connect(port)    print(f"Proxy público: {tunnel.public_url}")    return tunnel.public_urldef run_worker():    """Inicia worker completo."""    install_deps()    t = threading.Thread(target=run_proxy, daemon=True)    t.start()    time.sleep(2)    url = expose_ngrok()    print(f"Worker ativo! URL: {url}")    while True:        print(f"[{time.strftime('%H:%M:%S')}] Worker rodando, proxy em {url}")        time.sleep(300)# run_worker()